In [1]:
from datasets import load_dataset

ds = load_dataset("nominal-io/nasa-turbofan-degradation", split="train")
print(ds)
print(ds.features)
print("keys:", ds[0].keys())

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 21495 examples [00:00, 96874.00 examples/s]

Dataset({
    features: ['Unnamed: 0', 'engine', 'cycle', 'setting_1', 'setting_2', 'setting_3', '(Fan inlet temperature) (◦R)', '(LPC outlet temperature) (◦R)', '(HPC outlet temperature) (◦R)', '(LPT outlet temperature) (◦R)', '(Fan inlet Pressure) (psia)', '(bypass-duct pressure) (psia)', '(HPC outlet pressure) (psia)', '(Physical fan speed) (rpm)', '(Physical core speed) (rpm)', '(Engine pressure ratio(P50/P2)', '(HPC outlet Static pressure) (psia)', '(Ratio of fuel flow to Ps30) (pps/psia)', '(Corrected fan speed) (rpm)', '(Corrected core speed) (rpm)', '(Bypass Ratio) ', '(Burner fuel-air ratio)', '(Bleed Enthalpy)', '(Required fan speed)', '(Required fan conversion speed)', '(High-pressure turbines Cool air flow)', '(Low-pressure turbines Cool air flow)'],
    num_rows: 21495
})
{'Unnamed: 0': Value(dtype='int64', id=None), 'engine': Value(dtype='int64', id=None), 'cycle': Value(dtype='int64', id=None), 'setting_1': Value(dtype='float64', id=None), 'setting_2': Value(dtype='float

In [2]:
import pandas as pd
import numpy as np

df = ds.to_pandas()

id_col = "engine"
time_col = "cycle"

# 一般会有一个多余索引列
drop_cols = [c for c in ["Unnamed: 0", id_col, time_col] if c in df.columns]
feature_cols = [c for c in df.columns if c not in drop_cols]

print("n_rows:", len(df))
print("n_engines:", df[id_col].nunique())
print("cycle_max:", df[time_col].max())
print("n_features:", len(feature_cols))
print("first 5 feature cols:", feature_cols[:5])


n_rows: 21495
n_engines: 100
cycle_max: 362
n_features: 24
first 5 feature cols: ['setting_1', 'setting_2', 'setting_3', '(Fan inlet temperature) (◦R)', '(LPC outlet temperature) (◦R)']


In [3]:
df = df.sort_values([id_col, time_col]).reset_index(drop=True)

engine_ids = sorted(df[id_col].unique().tolist())
series_by_engine = {}
time_by_engine = {}

for eid in engine_ids:
    sub = df[df[id_col] == eid].sort_values(time_col)
    series_by_engine[eid] = sub[feature_cols].to_numpy(dtype=np.float32)  # [T, F]
    time_by_engine[eid] = sub[time_col].to_numpy(dtype=np.int64)          # [T]

print("num engines:", len(series_by_engine))
print("engine 1:", series_by_engine[1].shape, time_by_engine[1].shape, 
      "cycle range:", time_by_engine[1][0], "->", time_by_engine[1][-1])


num engines: 100
engine 1: (384, 24) (384,) cycle range: 1 -> 192


In [4]:
def make_windows(x_tf, t_arr, L=64, stride=16):
    # x_tf: [T, F], t_arr: [T]
    T, F = x_tf.shape
    windows = []
    times = []
    for end in range(L, T + 1, stride):
        start = end - L
        w = x_tf[start:end]             # [L, F]
        ts = int(t_arr[end - 1])        # 窗口末端 cycle 作为 timestamp
        windows.append(w)
        times.append(ts)
    return np.stack(windows), np.array(times)

L = 64
stride = 16
w1, t1 = make_windows(series_by_engine[1], time_by_engine[1], L=L, stride=stride)
print("windows for engine=1:", w1.shape, t1.shape)
print("first/last window timestamp:", t1[0], t1[-1])


windows for engine=1: (21, 64, 24) (21,)
first/last window timestamp: 32 192
